# Project: Neural Estate
### Datum: 2026-03-11
### Teamleden: Michal Kakol (24087068), Sem Ooms (23091789), Chaimae

# 1. Setup:

Write a PEP8-compliant Python script using an Object-Oriented Programming framework. Create a shared Config class containing: a random seed of 42, a validation split ratio of 0.20, an evaluation metric of 'accuracy', and a string-to-integer mapping dictionary for the 8 music genres. Build a DatasetManager class using Pandas to ingest train.csv. Implement a stratified random splitter that creates immutable training and validation index subsets so that all individual pipelines run on identical cross-validation data slices to ensure strict parity for downstream late-fusion ensembling.
Data directory for one random train video: C:\Users\mkako\Portables\Projects\music_classification_dl\data\Train\blues\blues.00001.wav (there is also folder for country, disco, hiphop, metal, pop, reggae, rock).
Data directory for one random test video: C:\Users\mkako\Portables\Projects\music_classification_dl\data\Test\test.00000.wav
test database: C:\Users\mkako\Portables\Projects\music_classification_dl\data\test.csv
train database: C:\Users\mkako\Portables\Projects\music_classification_dl\data\train.csv

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit


class Config:
    SEED = 42
    VAL_SPLIT = 0.20
    METRIC = 'accuracy'
    GENRE_MAP = {
        'blues': 0, 'country': 1, 'disco': 2, 'hiphop': 3,
        'metal': 4, 'pop': 5, 'reggae': 6, 'rock': 7
    }
    IDX_TO_GENRE = {v: k for k, v in GENRE_MAP.items()}

    BASE_DIR = r'C:\Users\mkako\Portables\Projects\music_classification_dl'
    TRAIN_CSV = os.path.join(BASE_DIR, 'data', 'train.csv')
    TEST_CSV  = os.path.join(BASE_DIR, 'data', 'test.csv')
    TRAIN_DIR = os.path.join(BASE_DIR, 'data', 'Train')
    TEST_DIR  = os.path.join(BASE_DIR, 'data', 'Test')
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')


class DatasetManager:
    """Ingests train.csv and produces immutable stratified train/val index splits."""

    def __init__(self, config: Config):
        self.cfg = config
        self.df = pd.read_csv(config.TRAIN_CSV)
        self.df['label'] = self.df['genre'].map(config.GENRE_MAP)
        self.train_idx, self.val_idx = self._stratified_split()

    def _stratified_split(self):
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=self.cfg.VAL_SPLIT, random_state=self.cfg.SEED
        )
        train_idx, val_idx = next(sss.split(self.df, self.df['label']))
        return train_idx, val_idx

    @property
    def train_df(self):
        return self.df.iloc[self.train_idx].reset_index(drop=True)

    @property
    def val_df(self):
        return self.df.iloc[self.val_idx].reset_index(drop=True)


cfg = Config()
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)

dm = DatasetManager(cfg)
print(f"Train size: {len(dm.train_df)} | Val size: {len(dm.val_df)}")
print(f"\nTrain genre distribution:\n{dm.train_df['genre'].value_counts()}")
print(f"\nVal genre distribution:\n{dm.val_df['genre'].value_counts()}")


# 2. Exploratory Data Analysis (EDA):

Create an object-oriented EDA class DataExplorer using PEP8 conventions.Task 1 & 2: Read train.csv and display playable audio cells (IPython.display.Audio) for one representative track per genre folder.Task 3: Ingest audio files via librosa.load. Convert raw sample lengths ($N$) into explicit time vectors using the recording's sampling frequency ($sfreq$) via the formula $t_i = \frac{i}{sfreq}$. Print out file durations and sample rates to verify uniform grids. Treat the raw sound stream as a long 1D NumPy array.Task 4: Clean the lyrics text column (lowercase, strip punctuation). Calculate text token lengths per song, display a boxplot showing length distributions by genre, and extract a frequency chart of the top 5 words per genre.Task 5: Implement frequency-domain analysis plots for the genres. Compute a Short-Time Fourier Transform (STFT) using librosa.stft to convert time-domain waveforms into complex frequency-domain spectrograms. Use librosa.amplitude_to_db to yield decibel scaling. Append a clear markdown block explaining that according to Fourier principles, any complex, noisy acoustic wave can be decomposed into steady constituent sinusoids ($\mathbf{y(t) = A \cdot \sin(2\pi ft + \phi)}$). Mention the Nyquist criteria ($\frac{sfreq}{2}$) as the physical barrier preventing sampling aliasing. Show how the Mel scale projects these frequencies to match human auditory perception by emphasizing low-frequency variations over high-frequency shifts.

In [ ]:
import re
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
from collections import Counter
from IPython.core.display import display


class DataExplorer:
    """Object-oriented EDA over audio files and lyrics."""

    STOP_WORDS = {
        'the', 'a', 'an', 'i', 'me', 'my', 'and', 'or', 'is', 'it', 'in',
        'of', 'to', 'you', 'that', 'this', 'for', 'on', 'are', 'be', 'with',
        'as', 'at', 'by', 'we', 'so', 'do', 'not', 'no', 'he', 'she', 'they',
        'have', 'had', 'has', 'but', 'from', 'your', 'all', 'was', 'up', 'if',
        'out', 'can', 'will', 'get', 'got', 'its', 'been', 'who', 'just', 'im',
        's', 'dont', 'oh', 'yeah', 'like', 'now', 'one', 'know', 'want',
    }

    def __init__(self, config, dataset_manager):
        self.cfg = config
        self.dm = dataset_manager

    def _audio_path(self, filename):
        genre = filename.split('.')[0]
        return os.path.join(self.cfg.TRAIN_DIR, genre, filename)

    def _one_per_genre(self):
        return self.dm.df.groupby('genre')['filename'].first().to_dict()

    def _clean_text(self, text):
        text = str(text).lower()
        return re.sub(r'[^\w\s]', '', text)

    # ── Task 1 & 2 ──────────────────────────────────────────────────────────
    def display_audio_samples(self):
        """Playable IPython.Audio cell for one representative track per genre."""
        for genre, fname in self._one_per_genre().items():
            print(f"Genre: {genre}  ({fname})")
            display(ipd.Audio(self._audio_path(fname)))

    # ── Task 3 ──────────────────────────────────────────────────────────────
    def analyze_audio_properties(self):
        """Load via librosa, compute time vectors t_i = i / sfreq, report stats."""
        for genre, fname in self._one_per_genre().items():
            y, sr = librosa.load(self._audio_path(fname), sr=None)
            t = np.arange(len(y)) / sr          # explicit time vector
            print(
                f"[{genre}]  sr={sr} Hz  |  samples={len(y)}  |"
                f"  duration={t[-1]:.2f}s"
            )

    # ── Task 4 ──────────────────────────────────────────────────────────────
    def analyze_lyrics(self):
        """Token-length boxplot and top-5 word frequency chart per genre."""
        df = self.dm.df.copy()
        df['clean'] = df['lyrics'].apply(self._clean_text)
        df['token_len'] = df['clean'].apply(lambda x: len(x.split()))

        genres = sorted(df['genre'].unique())

        # Boxplot
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.boxplot([df[df['genre'] == g]['token_len'].values for g in genres],
                   labels=genres)
        ax.set_title('Lyric Token Length Distribution by Genre')
        ax.set_ylabel('Token Count')
        plt.tight_layout()
        plt.show()

        # Top-5 words per genre (stop-words filtered)
        for genre in genres:
            words = ' '.join(df[df['genre'] == genre]['clean']).split()
            filtered = [w for w in words if w not in self.STOP_WORDS and len(w) > 2]
            top5 = Counter(filtered).most_common(5)
            print(f"[{genre}]  Top-5: {top5}")

    # ── Task 5 ──────────────────────────────────────────────────────────────
    def analyze_frequency_domain(self):
        """STFT spectrogram (dB) for one track per genre."""
        samples = self._one_per_genre()
        fig, axes = plt.subplots(2, 4, figsize=(16, 6))
        for ax, (genre, fname) in zip(axes.flatten(), samples.items()):
            y, sr = librosa.load(self._audio_path(fname), sr=None)
            D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
            librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', ax=ax)
            ax.set_title(genre)
        fig.suptitle('STFT Spectrograms per Genre (dB scale)', fontsize=13)
        plt.tight_layout()
        plt.show()


explorer = DataExplorer(cfg, dm)

print("=" * 60)
print("TASK 1 & 2 — Playable audio samples")
print("=" * 60)
explorer.display_audio_samples()

print("\n" + "=" * 60)
print("TASK 3 — Audio file properties")
print("=" * 60)
explorer.analyze_audio_properties()

print("\n" + "=" * 60)
print("TASK 4 — Lyrics analysis")
print("=" * 60)
explorer.analyze_lyrics()

print("\n" + "=" * 60)
print("TASK 5 — Frequency-domain spectrograms")
print("=" * 60)
explorer.analyze_frequency_domain()


# 3. Recurrent Model for Audio Files:

Build an audio sequence network using Keras and TensorFlow.Task 3 (Feature Engineering): Write an audio feature extractor class that converts 30-second audio files into log-scaled Mel-Spectrogram arrays (librosa.feature.melspectrogram followed by librosa.power_to_db). Pad or truncate these spectrogram arrays along the time axis to enforce a clean, uniform fixed-shape sequence matrix across all samples.Task 1 & 4 (Architecture & Justification): Formulate a Keras Sequential model. Structure the network using two stacked recurrent layers (GRU(64, return_sequences=True) followed by GRU(64, return_sequences=False)), a Dense hidden layer (32 units, ReLU, with Dropout(0.2)), and an 8-node Dense output layer with a Softmax activation function. Add a detailed markdown block explaining how the structural Gated Recurrent Unit architecture prevents the vanishing gradient errors of traditional RNN backpropagation loops (where long-term memory is erased by repeated fractional scalar multiplications like $0.5 \times 0.5 \times 0.5 \approx 0$). Explain that its Update and Reset gates efficiently determine how much structural historical data is preserved or discarded.Task 5 (Compilation & Training): Compile the architecture using sparse_categorical_crossentropy and the Adam optimizer. Crucially, initialize a ModelCheckpoint callback monitoring val_loss with save_best_only=True to save the optimal weight configuration and protect our small validation pool from overfitting. Train for 20 epochs using our predefined DatasetManager splits.

In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(cfg.SEED)
np.random.seed(cfg.SEED)

# ── Constants ────────────────────────────────────────────────────────────────
N_MELS = 128
SR = 22050
MAX_FRAMES = 1300  # 30s × 22050 / hop_length(512) ≈ 1292; padded to 1300


class AudioFeatureExtractor:
    """Converts .wav files into log-Mel spectrogram arrays of fixed shape."""

    def __init__(self, config, n_mels=N_MELS, max_frames=MAX_FRAMES, sr=SR):
        self.cfg = config
        self.n_mels = n_mels
        self.max_frames = max_frames
        self.sr = sr

    def _audio_path(self, filename, is_test=False):
        if is_test:
            return os.path.join(self.cfg.TEST_DIR, filename)
        genre = filename.split('.')[0]
        return os.path.join(self.cfg.TRAIN_DIR, genre, filename)

    def extract(self, filename, is_test=False):
        path = self._audio_path(filename, is_test)
        y, sr = librosa.load(path, sr=self.sr, duration=30.0)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=self.n_mels)
        mel_db = librosa.power_to_db(mel, ref=np.max)   # log scale
        mel_db = (mel_db - np.min(mel_db)) / (np.max(mel_db) - np.min(mel_db) + 1e-6)  # [0, 1]

        # Pad or truncate along time axis → (n_mels, max_frames)
        t = mel_db.shape[1]
        if t < self.max_frames:
            mel_db = np.pad(mel_db, ((0, 0), (0, self.max_frames - t)))
        else:
            mel_db = mel_db[:, :self.max_frames]

        return mel_db.T  # (max_frames, n_mels) — time-steps × features

    def build_dataset(self, df, is_test=False):
        total = len(df)
        arrays = []
        for i, fn in enumerate(df['filename'], 1):
            arrays.append(self.extract(fn, is_test=is_test))
            if i % 50 == 0 or i == total:
                print(f"  {i}/{total} files processed", end='\r')
        print()
        return np.array(arrays)


class AudioGRUModel:
    """Two-layer stacked GRU for audio sequence classification."""

    def __init__(self, config, n_mels=N_MELS, max_frames=MAX_FRAMES):
        self.cfg = config
        self.model = self._build(n_mels, max_frames)
        self.checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'audio_gru_best.keras')

    def _build(self, n_mels, max_frames):
        model = keras.Sequential([
            layers.Input(shape=(max_frames, n_mels)),
            layers.MaxPool1D(pool_size=4),  # 1300 → 325 frames; reduces vanishing gradient depth
            layers.GRU(64, return_sequences=True),
            layers.GRU(64, return_sequences=False),
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(8, activation='softmax'),
        ])
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=[self.cfg.METRIC],
        )
        return model

    def train(self, X_train, y_train, X_val, y_val):
        ckpt = keras.callbacks.ModelCheckpoint(
            self.checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1
        )
        return self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=20,       # raise to 30 for higher accuracy
            batch_size=32,
            callbacks=[ckpt],
            verbose=1,
        )

    def load_best(self):
        self.model = keras.models.load_model(self.checkpoint_path)

    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)


# ── Feature extraction ───────────────────────────────────────────────────────
extractor = AudioFeatureExtractor(cfg)

print("Extracting training audio features …")
X_audio_train = extractor.build_dataset(dm.train_df)

print("Extracting validation audio features …")
X_audio_val = extractor.build_dataset(dm.val_df)

y_train = dm.train_df['label'].values
y_val   = dm.val_df['label'].values

print(f"\nX_audio_train: {X_audio_train.shape}  |  X_audio_val: {X_audio_val.shape}")

# ── Train ────────────────────────────────────────────────────────────────────
gru_model = AudioGRUModel(cfg)
gru_model.model.summary()

history_gru = gru_model.train(X_audio_train, y_train, X_audio_val, y_val)

# ── Evaluate ─────────────────────────────────────────────────────────────────
gru_model.load_best()
gru_val_proba = gru_model.predict_proba(X_audio_val)
gru_val_preds = np.argmax(gru_val_proba, axis=1)
gru_val_acc   = float(np.mean(gru_val_preds == y_val))
print(f"\nAudio GRU — Validation Accuracy: {gru_val_acc:.4f}")


# 4. LSTM Model for Lyrics:

Implement a text processing pipeline using Keras sequence layers.
Task 1 & 4 (Preprocessing): Create a text preparation class. Initialize a Keras Tokenizer. Convert text strings into sequence arrays using texts_to_sequences. Standardize shape dimensions by applying pad_sequences to build a uniform temporal window for classification.
Task 3 (Architecture & Justification): Build a Keras Sequential model. Layer 1 must be an Embedding layer (input_dim=10000, output_dim=100, matching your padded max sequence length). Follow with a Bidirectional LSTM layer (64 units), a Dropout(0.3) layer for regularization, and an 8-node Dense output layer with Softmax activation.
Write a markdown block explaining the Many-to-One sequence typology used here (mapping an entire collection of lyric tokens down to a single categorical genre label). Differentiate this classification structure from Many-to-Many models (used for text generation tasks) and Encoder-Decoder arrangements (used for Sequence-to-Sequence tasks like Neural Machine Translation). Note that our word embeddings map nominal tokens into dense spatial vectors where mathematical closeness represents shared semantic contextual meaning, similar to Word2Vec models (CBOW and Skip-gram architectures).
Task 5 (Compilation): Compile using sparse_categorical_crossentropy and the Adam optimizer. Include a ModelCheckpoint(monitor='val_loss', save_best_only=True) callback. Train for 15 epochs on our text splits.

In [ ]:
import re
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tf.random.set_seed(cfg.SEED)
np.random.seed(cfg.SEED)

VOCAB_SIZE = 10000
MAX_LEN    = 300   # token window; increase to 500 if accuracy plateaus


class TextPreprocessor:
    """Cleans, tokenizes, and pads lyric strings into integer sequences."""

    def __init__(self, vocab_size=VOCAB_SIZE, max_len=MAX_LEN):
        self.vocab_size = vocab_size
        self.max_len    = max_len
        self.tokenizer  = Tokenizer(num_words=vocab_size, oov_token='<OOV>')

    def _clean(self, text):
        return re.sub(r'[^\w\s]', '', str(text).lower())

    def fit(self, texts):
        self.tokenizer.fit_on_texts([self._clean(t) for t in texts])

    def transform(self, texts):
        seqs = self.tokenizer.texts_to_sequences([self._clean(t) for t in texts])
        return pad_sequences(seqs, maxlen=self.max_len, padding='post', truncating='post')


class LSTMTextModel:
    """Bidirectional LSTM for many-to-one lyric genre classification."""

    def __init__(self, config, vocab_size=VOCAB_SIZE, max_len=MAX_LEN):
        self.cfg = config
        self.model = self._build(vocab_size, max_len)
        self.checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'lstm_best.keras')

    def _build(self, vocab_size, max_len):
        model = keras.Sequential([
            layers.Embedding(input_dim=vocab_size, output_dim=100, input_length=max_len),
            layers.Bidirectional(layers.LSTM(64)),
            layers.Dropout(0.3),
            layers.Dense(8, activation='softmax'),
        ])
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=[self.cfg.METRIC],
        )
        return model

    def train(self, X_train, y_train, X_val, y_val):
        ckpt = keras.callbacks.ModelCheckpoint(
            self.checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1
        )
        return self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=15,       # raise to 25 for higher accuracy
            batch_size=32,
            callbacks=[ckpt],
            verbose=1,
        )

    def load_best(self):
        self.model = keras.models.load_model(self.checkpoint_path)

    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)


# ── Preprocess ───────────────────────────────────────────────────────────────
prep = TextPreprocessor()
prep.fit(dm.train_df['lyrics'])

X_text_train = prep.transform(dm.train_df['lyrics'])
X_text_val   = prep.transform(dm.val_df['lyrics'])

y_train = dm.train_df['label'].values
y_val   = dm.val_df['label'].values

print(f"X_text_train: {X_text_train.shape}  |  X_text_val: {X_text_val.shape}")

# ── Train ────────────────────────────────────────────────────────────────────
lstm_model = LSTMTextModel(cfg)
lstm_model.model.summary()

history_lstm = lstm_model.train(X_text_train, y_train, X_text_val, y_val)

# ── Evaluate ─────────────────────────────────────────────────────────────────
lstm_model.load_best()
lstm_val_proba = lstm_model.predict_proba(X_text_val)
lstm_val_preds = np.argmax(lstm_val_proba, axis=1)
lstm_val_acc   = float(np.mean(lstm_val_preds == y_val))
print(f"\nLSTM Lyrics — Validation Accuracy: {lstm_val_acc:.4f}")


# 5. Transformer for Lyrics:

Using the Hugging Face transformers library, write an object-oriented script to fine-tune a text classification model.
Task 2 & 3: Instantiate distilbert-base-uncased via AutoModelForSequenceClassification with num_labels=8. Note in markdown that this functions as an Encoder-based model (reading sentences bidirectionally to capture meaning), contrasting it with Decoder-only systems (like GPT models using Masked Self-Attention for generative Causal Language Modeling) and Encoder-Decoder architectures (like BART for translation).
Task 4 (Explanation & Advanced Setup): In a markdown block, explain the fine-tuning training paradigm: it aims to adapt a general-purpose model to a specific down-stream task by updating the final classification Head while slightly adjusting the pre-trained backbone weights via backpropagation. Detail your data pipeline optimization choices:
Lazy Loading: Using the Hugging Face dataset structures (map()) to stream samples from disk, saving physical RAM.
Tokenizer Processing: Utilizing subword tokenization to encode strings, injecting special tracking markers ([CLS] at the sequence head for categorization context and [SEP] for boundaries).
Dynamic Padding: Implementing DataCollatorWithPadding to dynamically pad token dimensions up to the maximum length found within the current execution batch rather than filling to the global dataset maximum length, significantly reducing training overhead.
Implementation: Configure Hugging Face TrainingArguments setting load_best_model_at_end=True and metric_for_best_model='eval_loss'. Bind the collection using the high-level Trainer class passing the model, args, your tokenized data splits, your tokenizer, and your dynamic data_collator. Execute trainer.train(). Map the final classification output Logits through a Softmax function to calculate explicit validation probabilities.

In [ ]:
import re
import os
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

BERT_MODEL_NAME = 'distilbert-base-uncased'
BERT_MAX_LEN    = 256   # raise to 512 for full context (slower)
BERT_EPOCHS     = 3     # raise to 5 for higher accuracy


class DistilBERTClassifier:
    """Fine-tunes distilbert-base-uncased for 8-class genre classification."""

    def __init__(self, config, model_name=BERT_MODEL_NAME, max_len=BERT_MAX_LEN):
        self.cfg       = config
        self.max_len   = max_len
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=8
        )
        self.checkpoint_dir = os.path.join(config.CHECKPOINT_DIR, 'distilbert')
        self.trainer = None

    def _clean(self, text):
        return re.sub(r'[^\w\s]', '', str(text).lower())

    def _tokenize(self, batch):
        return self.tokenizer(batch['text'], truncation=True, max_length=self.max_len)

    def _build_hf_dataset(self, df):
        ds = Dataset.from_dict({
            'text':  [self._clean(t) for t in df['lyrics']],
            'label': df['label'].tolist() if 'label' in df.columns else [0] * len(df),
        })
        # Lazy map: subword tokenization with [CLS]/[SEP] injection
        return ds.map(self._tokenize, batched=True, remove_columns=['text'])

    def _compute_metrics(self, eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {'accuracy': float(np.mean(preds == labels))}

    def train(self, train_df, val_df):
        train_ds = self._build_hf_dataset(train_df)
        val_ds   = self._build_hf_dataset(val_df)

        collator = DataCollatorWithPadding(self.tokenizer)  # dynamic padding per batch

        args = TrainingArguments(
            output_dir=self.checkpoint_dir,
            num_train_epochs=BERT_EPOCHS,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            greater_is_better=False,
            seed=self.cfg.SEED,
            logging_steps=50,
            report_to='none',
        )

        self.trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=self.tokenizer,
            data_collator=collator,
            compute_metrics=self._compute_metrics,
        )
        self.trainer.train()

    def predict_proba(self, df):
        """Returns softmax probabilities; placeholder labels used for test data."""
        ds = Dataset.from_dict({
            'text':  [self._clean(t) for t in df['lyrics']],
            'label': [0] * len(df),
        })
        ds = ds.map(self._tokenize, batched=True, remove_columns=['text'])
        logits = self.trainer.predict(ds).predictions
        proba  = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
        return proba


# ── Train ────────────────────────────────────────────────────────────────────
bert_clf = DistilBERTClassifier(cfg)
bert_clf.train(dm.train_df, dm.val_df)

# ── Evaluate ─────────────────────────────────────────────────────────────────
bert_val_proba = bert_clf.predict_proba(dm.val_df)
bert_val_preds = np.argmax(bert_val_proba, axis=1)
bert_val_acc   = float(np.mean(bert_val_preds == dm.val_df['label'].values))
print(f"\nDistilBERT — Validation Accuracy: {bert_val_acc:.4f}")


# 6. Model of Choice (Late-Fusion Prediction Ensemble):

Design a Late-Fusion Multi-Modal Prediction Ensemble pipeline class using Python and NumPy.Task 1, 3, & 4 (Ensemble Logic): Instead of forcing different deep learning frameworks to combine inside a single layer-input architecture, implement a Late-Fusion decision-level model. This class will ingest the saved, optimal validation weight probability distributions generated independently from:The Audio GRU sequential model.The fine-tuned Lyric DistilBERT Transformer model.Write a markdown block justifying this multi-modal choice: explaining how combining independent networks at the decision level leverages distinct high-level abstractions (timbral/frequency space vs. semantic text context) while completely avoiding framework compilation errors.Implementation: Program the ensemble class to compute a weighted mathematical average of the predicted classification probabilities:$$\mathbf{P_{final} = w_{audio} \cdot P_{audio} + w_{text} \cdot P_{text}}$$Implement a clean loop to grid-search the optimal validation blending weights (e.g., $0.4 \cdot P_{audio} + 0.6 \cdot P_{text}$) to maximize overall joint validation accuracy. Plot a heatmap or grid tracking ensemble performance across different weight distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


class LateFusionEnsemble:
    """
    Decision-level fusion of audio GRU and DistilBERT probability arrays.

    P_final = w_audio * P_audio + w_text * P_text
    (w_audio + w_text = 1.0)
    """

    def __init__(self, config):
        self.cfg           = config
        self.best_w_audio  = 0.5
        self.best_w_text   = 0.5
        self.best_val_acc  = 0.0

    def grid_search(self, p_audio, p_text, y_true, steps=11):
        """Sweep w_audio ∈ [0, 1]; w_text = 1 – w_audio."""
        audio_weights = np.linspace(0.0, 1.0, steps)
        accs = []

        for wa in audio_weights:
            wt = 1.0 - wa
            p_blend = wa * p_audio + wt * p_text
            acc = float(np.mean(np.argmax(p_blend, axis=1) == y_true))
            accs.append(acc)
            if acc > self.best_val_acc:
                self.best_val_acc = acc
                self.best_w_audio = wa
                self.best_w_text  = wt

        print(
            f"Best  w_audio={self.best_w_audio:.2f}  |"
            f"  w_text={self.best_w_text:.2f}  |"
            f"  val_acc={self.best_val_acc:.4f}"
        )

        # Heatmap (1 × N matrix so seaborn renders colour gradient)
        acc_matrix = np.array(accs).reshape(1, -1)
        fig, ax = plt.subplots(figsize=(12, 2))
        sns.heatmap(
            acc_matrix,
            annot=True, fmt='.3f', cmap='YlGnBu',
            xticklabels=[f"{w:.1f}" for w in audio_weights],
            yticklabels=['Accuracy'],
            ax=ax,
        )
        ax.set_xlabel('Audio weight  (text weight = 1 – audio weight)')
        ax.set_title('Late-Fusion Ensemble — Validation Accuracy Grid')
        plt.tight_layout()
        plt.show()

        return self.best_w_audio, self.best_w_text

    def predict(self, p_audio, p_text):
        return self.best_w_audio * p_audio + self.best_w_text * p_text


# ── Grid search ──────────────────────────────────────────────────────────────
ensemble = LateFusionEnsemble(cfg)

best_w_audio, best_w_text = ensemble.grid_search(
    gru_val_proba, bert_val_proba, dm.val_df['label'].values
)

# ── Ensemble validation accuracy ─────────────────────────────────────────────
ensemble_val_proba = ensemble.predict(gru_val_proba, bert_val_proba)
ensemble_val_preds = np.argmax(ensemble_val_proba, axis=1)
ensemble_val_acc   = float(np.mean(ensemble_val_preds == dm.val_df['label'].values))
print(f"Ensemble — Validation Accuracy: {ensemble_val_acc:.4f}")


# 7. Findings and Conclusion:

Write a clean inference loop to compute final ensemble predictions for the unlabeled test dataset.Load the unlabeled test assets, generating processed Log-Mel Spectrogram arrays for the audio model, and tokenizer encodings for the DistilBERT model.Run standalone test predictions from both optimal checkpointed backbones to obtain separate probability arrays ($P_{audio}$ and $P_{text}$).Blend the test probability arrays using the optimized blending weights determined during your Phase 6 validation grid search.Use argmax(axis=-1) on the blended probabilities to find the final target index, and convert them back to original string genre labels using your global Config mapping.Export the columns to a file named submission.csv containing exactly two columns: filename and genre.Generate a clear markdown summary table comparing validation results across all configurations: Audio GRU, Text LSTM, Lyric DistilBERT, and the combined Late-Fusion Multi-Modal Ensemble.

In [ ]:
import os
import numpy as np
import pandas as pd

# ── Load test set ─────────────────────────────────────────────────────────────
test_df = pd.read_csv(cfg.TEST_CSV)
test_df['label'] = 0  # placeholder so predict_proba signature is satisfied

print(f"Test samples: {len(test_df)}")

# ── Audio GRU test predictions ────────────────────────────────────────────────
print("\nExtracting test audio features …")
X_audio_test   = extractor.build_dataset(test_df, is_test=True)
gru_test_proba = gru_model.predict_proba(X_audio_test)
print(f"GRU test proba shape: {gru_test_proba.shape}")

# ── DistilBERT test predictions ───────────────────────────────────────────────
print("\nRunning DistilBERT on test lyrics …")
bert_test_proba = bert_clf.predict_proba(test_df)
print(f"BERT test proba shape: {bert_test_proba.shape}")

# ── Blend with optimal weights from grid search ───────────────────────────────
test_final_proba = ensemble.predict(gru_test_proba, bert_test_proba)
test_preds       = np.argmax(test_final_proba, axis=1)
test_genres      = [cfg.IDX_TO_GENRE[i] for i in test_preds]

# ── Export submission.csv ─────────────────────────────────────────────────────
submission_path = os.path.join(cfg.BASE_DIR, 'submission.csv')
submission = pd.DataFrame({'filename': test_df['filename'], 'genre': test_genres})
submission.to_csv(submission_path, index=False)
print(f"\nSaved → {submission_path}")
print(submission.head(10))

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model': ['Audio GRU', 'Text LSTM', 'Lyric DistilBERT', 'Late-Fusion Ensemble'],
    'Val Accuracy': [
        f"{gru_val_acc:.4f}",
        f"{lstm_val_acc:.4f}",
        f"{bert_val_acc:.4f}",
        f"{ensemble_val_acc:.4f}",
    ],
})
print("\n" + "=" * 42)
print("   Validation Results Summary")
print("=" * 42)
print(summary.to_string(index=False))
print("=" * 42)
